# Serializing and Deserializing AutoGen Components

This notebook demonstrates how to serialize and deserialize AutoGen components (termination conditions, agents, multimodal agents, and teams) using the Component configuration class. This is useful for debugging, visualizing, and sharing your work with others.

> **Warning**
>
> ONLY LOAD COMPONENTS FROM TRUSTED SOURCES. Serialized components may include executable code.

## Outline
1. Import Required Libraries and Setup
2. Serialize and Deserialize Termination Conditions
3. Serialize and Deserialize Agents
4. Serialize and Deserialize MultimodalWebSurfer Agent
5. Serialize and Deserialize Teams
6. (Optional) Save and Load Serialized Components to/from JSON Files

In [ ]:
# 1. Import Required Libraries and Setup
import json
from autogen_agentchat.conditions import MaxMessageTermination, StopMessageTermination
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.agents.web_surfer import MultimodalWebSurfer

# (Optional) Set up API keys or other configuration here if needed
# For demonstration, we use model="gpt-4o" and do not require an API key for serialization.

## 2. Serialize and Deserialize Termination Conditions

We will define termination conditions, serialize them to JSON, and demonstrate deserialization.

In [ ]:
# Define and serialize termination conditions
max_termination = MaxMessageTermination(5)
stop_termination = StopMessageTermination()
or_termination = max_termination | stop_termination

# Serialize to config (declarative spec)
or_term_config = or_termination.dump_component()
print("Serialized OrTerminationCondition (JSON):")
print(or_term_config.model_dump_json())

# Deserialize from config
new_or_termination = or_termination.load_component(or_term_config)
print("Deserialized OrTerminationCondition:", new_or_termination)

## 3. Serialize and Deserialize Agents

We will define agent objects, serialize them to JSON, and demonstrate deserialization.

In [ ]:
# Create model client (no API key needed for serialization)
model_client = OpenAIChatCompletionClient(model="gpt-4o")

# Create agents
agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    # tools=[], # serializing tools is not yet supported
    system_message="Use tools to solve tasks.",
)
user_proxy = UserProxyAgent(name="user")

# Serialize and print UserProxyAgent
user_proxy_config = user_proxy.dump_component()
print("Serialized UserProxyAgent (JSON):")
print(user_proxy_config.model_dump_json())

# Deserialize UserProxyAgent
up_new = user_proxy.load_component(user_proxy_config)
print("Deserialized UserProxyAgent:", up_new)

# Serialize and print AssistantAgent
agent_config = agent.dump_component()
print("Serialized AssistantAgent (JSON):")
print(agent_config.model_dump_json())

# Deserialize AssistantAgent
agent_new = agent.load_component(agent_config)
print("Deserialized AssistantAgent:", agent_new)

## 4. Serialize and Deserialize MultimodalWebSurfer Agent

We will define a MultimodalWebSurfer agent, serialize it to JSON, and demonstrate deserialization.

In [ ]:
# Create and serialize MultimodalWebSurfer agent
web_surfer = MultimodalWebSurfer(
    name="web_surfer",
    model_client=model_client,
    headless=False,
)

web_surfer_config = web_surfer.dump_component()
print("Serialized MultimodalWebSurfer (JSON):")
print(web_surfer_config.model_dump_json())

# Deserialize MultimodalWebSurfer
web_surfer_new = web_surfer.load_component(web_surfer_config)
print("Deserialized MultimodalWebSurfer:", web_surfer_new)

## 5. Serialize and Deserialize Teams

We will define a team (RoundRobinGroupChat), serialize it to JSON, and demonstrate deserialization.

In [ ]:
# Create a team with agent and termination condition
team = RoundRobinGroupChat(
    participants=[agent],
    termination_condition=MaxMessageTermination(2)
)

team_config = team.dump_component()
print("Serialized RoundRobinGroupChat (JSON):")
print(team_config.model_dump_json())

# Deserialize team
team_new = team.load_component(team_config)
print("Deserialized RoundRobinGroupChat:", team_new)

# Clean up model client
await model_client.close()

## 6. (Optional) Save and Load Serialized Components to/from JSON Files

You can save serialized component configurations to a JSON file and load them back using Python's built-in file I/O and `json` module.

In [ ]:
# Example: Save and load a serialized agent config
filename = "assistant_agent_config.json"

# Save to file
with open(filename, "w") as f:
    f.write(agent_config.model_dump_json())
print(f"Serialized AssistantAgent config saved to {filename}")

# Load from file
with open(filename, "r") as f:
    loaded_json = f.read()
    loaded_dict = json.loads(loaded_json)

# Reconstruct the component from loaded dict
agent_loaded = agent.load_component(agent_config.__class__(**loaded_dict))
print("Loaded AssistantAgent from file:", agent_loaded)